# FastAPI와 Slack 웹훅 (Echo 봇)
* 목표: Slack 메시지를 받아 **그대로 다시 보내기** (AI 없음)

```text
Slack에서 @봇 안녕
    → FastAPI /slack/events
    → Slack으로 "[echo] 안녕" 답장
```

오늘 할 일:
1. echo용 FastAPI 앱(`slack_app.py`) 만들기
2. 앱 + ngrok 실행하고 Event Subscription에 URL 등록하기
3. Slack에서 멘션해서 echo 되는지 확인하기



---
* `day17` 커널
* `.env`에 이미 있어야 할 것:

```text
SLACK_BOT_TOKEN=xoxb-...
SLACK_SIGNING_SECRET=...
```

* Slack App에 봇 스코프 `chat:write`, `app_mentions:read` 가 있고, 워크스페이스에 Install 되어 있다고 가정합니다. (16.3에서 만든 앱 재사용 가능)



In [1]:
!pip install fastapi uvicorn slack_sdk python-dotenv -q

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os

# 17일차 → SK Autonomous R&D → 2026_AI/.env
load_dotenv(Path.cwd().parents[1] / '.env')

token = os.getenv('MY_SLACK_TOKEN') or ''
secret = os.getenv('MY_SLACK_SIGNING_SECRET') or ''

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)
print('BOT_TOKEN set     :', bool(token))
print('SIGNING_SECRET set:', bool(secret))
print('BOT TOKEN:', token[:10] if token else '(missing)')
print('SIGNING SECRET:', secret[:8] if secret else '(missing)')



WORKDIR : /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/17일차
BOT_TOKEN set     : True
SIGNING_SECRET set: True


NameError: name 'SLACK_BOT_TOKEN' is not defined

---
## 1. Echo FastAPI 앱 만들기
* POST로 온 Slack 이벤트에서 텍스트를 읽고
* `chat.postMessage`로 `[echo] {내용}` 을 같은 채널(스레드)에 다시 보냅니다.
* LLM 호출 없음.

아래 셀을 실행해 `slack_app.py`를 저장하세요.



In [10]:
# AI 없는 echo 봇 — slack_app.py 생성

slack_app_py = '''
import hashlib
import hmac
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from fastapi import FastAPI, Header, HTTPException, Request
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError

load_dotenv()
load_dotenv(Path(__file__).resolve().parent.parent / ".env")

SLACK_BOT_TOKEN = os.environ["SLACK_BOT_TOKEN"]
SLACK_SIGNING_SECRET = os.environ["SLACK_SIGNING_SECRET"]

client = WebClient(token=SLACK_BOT_TOKEN)
app = FastAPI(title="Slack Echo Bot")
_seen: set[str] = set()


def verify_slack_signature(body: bytes, timestamp: str | None, signature: str | None) -> None:
    if not timestamp or not signature:
        raise HTTPException(status_code=401, detail="missing signature headers")
    if abs(time.time() - int(timestamp)) > 60 * 5:
        raise HTTPException(status_code=401, detail="stale request")
    base = f"v0:{timestamp}:{body.decode('utf-8')}"
    digest = hmac.new(
        SLACK_SIGNING_SECRET.encode(),
        base.encode(),
        hashlib.sha256,
    ).hexdigest()
    if not hmac.compare_digest(f"v0={digest}", signature):
        raise HTTPException(status_code=401, detail="invalid signature")


@app.get("/health")
def health():
    return {"ok": True}


@app.post("/slack/events")
async def slack_events(
    request: Request,
    x_slack_signature: str | None = Header(default=None),
    x_slack_request_timestamp: str | None = Header(default=None),
):
    body = await request.body()
    verify_slack_signature(body, x_slack_request_timestamp, x_slack_signature)
    payload = await request.json()

    # Slack Event Subscriptions URL 검증
    if payload.get("type") == "url_verification":
        return {"challenge": payload.get("challenge")}

    if payload.get("type") != "event_callback":
        return {"ok": True}

    event_id = payload.get("event_id")
    if event_id:
        if event_id in _seen:
            return {"ok": True}
        _seen.add(event_id)

    event = payload.get("event") or {}
    # 봇 메시지는 무시 (안 그러면 자기 echo에 또 반응)
    if event.get("bot_id") or event.get("subtype") == "bot_message":
        return {"ok": True}

    text = (event.get("text") or "").strip()
    channel = event.get("channel")
    if not text or not channel:
        return {"ok": True}

    # app_mention 이면 "<@U...> 내용" 에서 내용만 사용
    if event.get("type") == "app_mention":
        parts = text.split(maxsplit=1)
        text = parts[1] if len(parts) > 1 else text

    echo = f"[echo] {text}"
    print("받은 메시지:", text)
    try:
        # thread_ts를 넣지 않으면 채널에 일반 메시지로 올라갑니다
        client.chat_postMessage(channel=channel, text=echo)
    except SlackApiError as e:
        print("postMessage 실패:", e.response.get("error"))

    return {"ok": True}
'''

path = WORKDIR / 'slack_app.py'
path.write_text(slack_app_py.strip() + '\n', encoding='utf-8')
print('저장:', path)



저장: /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/17일차/slack_app.py


앱이 하는 일 요약:

| 요청 | 동작 |
|------|------|
| `url_verification` | `challenge` 그대로 반환 (URL 등록용) |
| `event_callback` + `app_mention` | 받은 문구를 `[echo] ...` 로 다시 전송 |
| 봇 자신의 메시지 | 무시 (무한 루프 방지) |



---
## 2. 앱 실행 → ngrok → Event Subscription (자세히)

**서버가 켜진 상태**에서 Slack에 URL을 넣어야 Verified가 됩니다.

### Step A. FastAPI 서버 켜기 (터미널 1)

```bash
conda activate day17
cd 17일차_실습
uvicorn slack_app:app --reload --port 8000
```

성공하면 대략 이렇게 보입니다.

```text
Uvicorn running on http://127.0.0.1:8000
```

브라우저에서 http://127.0.0.1:8000/health 를 열면 `{"ok":true}` 가 보이면 좋습니다.

> 이 터미널은 끄지 말고 그대로 둡니다.

---

### Step B. ngrok 설치 (처음 한 번)

로컬 `8000` 포트는 인터넷에서 안 보이므로, Slack이 우리 PC로 POST를 보내게 **공개 HTTPS 주소**가 필요합니다.

1. https://ngrok.com/download 에서 Windows용 설치 (또는 `winget install ngrok`)
2. ngrok 가입 후 대시보드의 authtoken 등록:

```bash
ngrok config add-authtoken <본인_토큰>
```

---

### Step C. ngrok으로 8000 포트 열기 (터미널 2)

**새 터미널**을 열고:

```bash
ngrok http 8000
```

화면에 비슷한 줄이 나옵니다.

```text
Forwarding    https://abcd-12-34-56.ngrok-free.app -> http://127.0.0.1:8000
```

* 여기서 **`https://....ngrok-free.app`** 부분만 복사합니다. (`http` 말고 `https`)
* ngrok을 끄거나 재시작하면 주소가 **바뀝니다**. 바뀌면 Slack URL도 다시 넣어야 합니다.

> 터미널 1(uvicorn)과 터미널 2(ngrok) **둘 다 켜 둔 상태**로 다음으로 갑니다.

---

### Step D. Slack Event Subscriptions에 URL 등록

1. 브라우저에서 https://api.slack.com/apps 접속
2. 사용할 **App** 클릭
3. 왼쪽 메뉴 **Event Subscriptions**
4. **Enable Events** 를 On
5. **Request URL** 칸에 아래처럼 입력 (본인 ngrok 주소로 바꿔서)

```text
https://abcd-12-34-56.ngrok-free.app/slack/events
```

주의:
* 끝 경로가 반드시 `/slack/events`
* `uvicorn` + `ngrok` 이 둘 다 살아 있어야 함

6. 잠시 기다리면 URL 옆에 **Verified** ✔ 가 뜹니다.

Verified가 안 되면 흔한 원인:
* uvicorn이 안 켜져 있음
* ngrok이 다른 포트를 보고 있음 (`8000`인지 확인)
* URL 오타 / `/slack/events` 빠짐
* Signing Secret이 `.env`와 Slack App의 값이 다름

---

### Step E. 구독할 이벤트 선택

같은 Event Subscriptions 페이지 아래 **Subscribe to bot events** 에서:

* `app_mention` 추가 (채널에서 `@봇` 했을 때)

필요하면:
* `message.im` (DM)

맨 아래 **Save Changes** 클릭.

워크스페이스에 앱을 다시 설치하라는 안내가 뜨면 **reinstall** 합니다.

---

### Step F. 채널에 봇 초대

테스트할 Slack 채널에서:

```text
/invite @봇이름
```



---
## 3. Slack에서 확인하기

채널에 입력:

```text
@봇이름 안녕하세요 echo 테스트
```

기대한 결과:
* 같은 스레드(또는 채널)에  
  `[echo] 안녕하세요 echo 테스트`  
  비슷한 메시지가 돌아옴
* uvicorn 터미널에 `받은 메시지: ...` 로그가 찍힘

안 되면 체크:
1. Event Subscriptions가 Verified 인가?
2. `app_mention` 구독 + Save 했는가?
3. `/invite`로 봇을 채널에 넣었는가?
4. ngrok URL이 바뀌지 않았는가?
5. 봇 토큰에 `chat:write` 권한이 있는가?



---
## 정리
1. `slack_app.py` — 받은 말을 `[echo]`로 되돌려 줌 (AI 없음)
2. `uvicorn` + `ngrok` → Slack Request URL Verified
3. `@봇` 멘션으로 왕복 확인

다음 단계에서 LLM/Agent를 붙이려면, echo 하던 `chat.postMessage` 자리에 답변 생성 코드를 넣으면 됩니다.

